In [1]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import random

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [3]:
# Dataset statistics before tokenization
emotions_unique = df["emotion"].unique()
print("Dataset statistics before tokenization for each unique emotion:")
print()
for emotion in emotions_unique:
    subset = df[df["emotion"] == emotion]
    N = len(subset["emotion"])
    print(f"{emotion} has {N} occurences, {round(N/len(df["emotion"]) * 100, 2)}%")
    average_length = subset["content"].str.len().mean()
    print(f"{emotion} has an average length of {average_length:.2f} characters")
    print()


Dataset statistics before tokenization for each unique emotion:

joy has 1092 occurences, 14.53%
joy has an average length of 103.14 characters

fear has 1076 occurences, 14.32%
fear has an average length of 123.96 characters

anger has 1079 occurences, 14.36%
anger has an average length of 130.65 characters

sadness has 1082 occurences, 14.4%
sadness has an average length of 103.93 characters

disgust has 1066 occurences, 14.18%
disgust has an average length of 114.61 characters

shame has 1071 occurences, 14.25%
shame has an average length of 116.73 characters

guilt has 1050 occurences, 13.97%
guilt has an average length of 124.89 characters



In [4]:
# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


In [5]:
df.head(5)
# print avarage token length of sentence here.

,emotion,content,tokens
0,joy,On days when I feel close to my partner and ot...,"[On, days, when, I, feel, close, to, my, partn..."
1,fear,Every time I imagine that someone I love or I ...,"[Every, time, I, imagine, that, someone, I, lo..."
2,anger,When I had been obviously unjustly treated and...,"[When, I, had, been, obviously, unjustly, trea..."
3,sadness,When I think about the short time that we live...,"[When, I, think, about, the, short, time, that..."
4,disgust,At a gathering I found myself involuntarily si...,"[At, a, gathering, I, found, myself, involunta..."


In [6]:
# Representation 2: Bag of words
vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)

bag_matrix = vectorizer.fit_transform(df["tokens"].tolist())

bag_df = pd.DataFrame(bag_matrix.toarray(),
                      columns=vectorizer.get_feature_names_out(),
                      index=df.index
)

print(bag_df.head(5))

   !  $  %  &  '  ''  'd  'll  'm  're  ...  yr.  yrs  yukky  zero  \
0  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
1  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
2  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
3  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
4  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   

   zig-zagging  zip  zipper  zombies  zone  zoophiliac  
0            0    0       0        0     0           0  
1            0    0       0        0     0           0  
2            0    0       0        0     0           0  
3            0    0       0        0     0           0  
4            0    0       0        0     0           0  

[5 rows x 10173 columns]


In [7]:
# Representation 3: Bigrams

bigrams = []
for sentence in df["tokens"]:
    bigrams_per_sentence = []
    for i in range(len(sentence) - 1):
        bigram = sentence[i] + " " + sentence[i+1]
        bigrams_per_sentence.append(bigram)
    bigrams.append(bigrams_per_sentence)
df["bigrams"] = bigrams

In [8]:
# Representation 4: Trigrams

trigrams = []
for sentence in df["tokens"]:
    trigrams_per_sentence = []
    for i in range(len(sentence) - 2):
        trigram = sentence[i] + " " + sentence[i+1] + " " + sentence[i+2]
        trigrams_per_sentence.append(trigram)
    trigrams.append(trigrams_per_sentence)
df["trigrams"] = trigrams

In [9]:
print(df["trigrams"][0])

['On days when', 'days when I', 'when I feel', 'I feel close', 'feel close to', 'close to my', 'to my partner', 'my partner and', 'partner and other', 'and other friends', 'other friends .', 'friends . When', '. When I', 'When I feel', 'I feel at', 'feel at peace', 'at peace with', 'peace with myself', 'with myself and', 'myself and also', 'and also experience', 'also experience a', 'experience a close', 'a close contact', 'close contact with', 'contact with people', 'with people whom', 'people whom I', 'whom I regard', 'I regard greatly', 'regard greatly .']


In [10]:
# Architecture 1: Log-Linear
import itertools
nltk.download("punkt_tab")

corpus = " ".join(df["content"].fillna("").astype(str))

vocab = sorted(set(word_tokenize(corpus)))
token_to_id = {token: i for i, token in enumerate(vocab)}

class Encoder:
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

# for seed in my_seeds:  
#     my_cross_validation_sets.append(
#         cross_validation_split(
#             # we pair the texts with their labels
#             pairs=list(zip(dataset["train"]["text"], dataset["train"]["label"])), 
#             k=num_folds,
#             seed=seed
#         )
#     )

class TokenIndicator(Encoder):
    def __init__(self, token_to_id):
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.token_to_id)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self), dtype=np.float32)

        for token in set(word_tokenize(x)):
            token_id = self.token_to_id.get(token)
            if token_id is not None:
                vector[token_id] = 1

        return vector
    
token_indicator_encoder = TokenIndicator(token_to_id)
doc_repr = token_indicator_encoder(corpus)

print(len(doc_repr))
print(doc_repr.sum())

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/ianhutcheson/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


10176
10176.0


In [11]:
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [12]:
# Architecture 2: Single hidden layer MLP
import smlp

class CountVectorizerEncoder(Encoder):
    def __init__(self, vectorizer):
        self.vectorizer = vectorizer

    def __len__(self):
        return len(self.vectorizer.vocabulary_)

    def __call__(self, x: str) -> np.ndarray:
        return self.vectorizer.transform([x]).toarray().ravel()

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)

# Settings for encoders for token indicator, BOW and unigrams + trigrams (capped due to memory issues).
encoder_configs = {
    "token-indicator": {
        "binary": True,
        "ngram_range": (1, 1),
    },
    "bag-of-words": {
        "binary": False,
        "ngram_range": (1, 1),
    },
    "bigrams-and-trigrams": {
        "binary": False,
        "ngram_range": (1, 2),
        "max_features": 20000
    },
}

results = {name: [] for name in encoder_configs}
mean_accuracy = {}
# For each encoder run the model with hidden dim of 128 with cross validation.
for encoder_name, config in encoder_configs.items():
    for fold in cv:
        print(encoder_name)
        training = cv[fold]["training"]
        validation = cv[fold]["validation"]

        vectorizer = CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            **config,
        )
        print("Fitting vectorizer")
        vectorizer.fit([content for content, label in training])
        print("Vocabulary size:", len(vectorizer.vocabulary_))
        print("Training MLP")
        encoder = CountVectorizerEncoder(vectorizer)

        metrics, best_epoch, best_pred = smlp.train_neural_network_model(
            training,
            validation,
            num_classes=len(labels),
            encoder=encoder,
            save_to=f"{results_folder}/mlp_{encoder_name}_{fold}_test",
            num_epochs=10,
        )

        best_val_acc_this_fold = max(epoch_data["val_metrics"]["accuracy"] for epoch_data in metrics)
        mean_accuracy[f"{encoder_name}"] = np.mean(best_val_acc_this_fold)
print(mean_accuracy)


token-indicator
Fitting vectorizer
Vocabulary size: 8343
Training MLP
Transforming documents...
Document transformation done! Time taken: 1.867 seconds
Dataloader done! Time taken: 0.279 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

/home/ianhutcheson/Documents/NLP-Classification-Project/Code/smlp.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/home/ianhutcheson/Documents/NLP-Classification-Project/Code/smlp.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()
/home/ianhutcheson/miniconda3/envs/

KeyboardInterrupt: 

In [12]:
# Architecture 3: Modified MLP - 2 hidden layers

import smlp_mod
class CountVectorizerEncoder(Encoder):
    def __init__(self, vectorizer):
        self.vectorizer = vectorizer

    def __len__(self):
        return len(self.vectorizer.vocabulary_)

    def __call__(self, x: str) -> np.ndarray:
        return self.vectorizer.transform([x]).toarray().ravel()

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)

encoder_configs = {
    "token-indicator": {
        "binary": True,
        "ngram_range": (1, 1),
    },
    "bag-of-words": {
        "binary": False,
        "ngram_range": (1, 1),
    },
    "bigrams-and-trigrams": {
        "binary": False,
        "ngram_range": (1, 2),
        "max_features": 75000
    },
}

results = {name: [] for name in encoder_configs}

encoder_accuracy = {}
for encoder_name, config in encoder_configs.items():
    mean_accuracy = []
    for fold in cv:
        print(encoder_name)
        training = cv[fold]["training"]
        validation = cv[fold]["validation"]

        vectorizer = CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            **config,
        )
        vectorizer.fit([content for content, label in training])
        print("Vocabulary size:", len(vectorizer.vocabulary_))
        encoder = CountVectorizerEncoder(vectorizer)

        metrics, best_epoch, best_pred = smlp_mod.train_neural_network_model(
            training,
            validation,
            num_classes=len(labels),
            encoder=encoder,
            save_to=f"{results_folder}/mlp_{encoder_name}_{fold}_test",
            num_epochs=10,
        )

        results[encoder_name].append(
            {
                "metrics": metrics,
                "best_epoch": best_epoch,
                "best_pred": best_pred,
            }
        )

        best_val_acc_this_fold = max(epoch_data["val_metrics"]["accuracy"] for epoch_data in metrics)
        mean_accuracy.append(best_val_acc_this_fold)
    encoder_accuracy[f"{encoder_name}"] = np.mean(mean_accuracy)



token-indicator
Vocabulary size: 8343
Transforming documents...
Document transformation done! Time taken: 1.780 seconds
Dataloader done! Time taken: 0.261 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

/home/ianhutcheson/Documents/NLP-Classification-Project/Code/smlp_mod.py:75: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/home/ianhutcheson/Documents/NLP-Classification-Project/Code/smlp_mod.py:78: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()
/home/ianhutcheson/minicond

Epoch 01 | train loss: 1.6641 | train acc: 0.3488 | val loss: 1.3095 | val acc: 0.5312
Epoch 02 | train loss: 1.1536 | train acc: 0.5960 | val loss: 1.2428 | val acc: 0.5499
Epoch 03 | train loss: 0.9187 | train acc: 0.6833 | val loss: 1.2615 | val acc: 0.5505
Epoch 04 | train loss: 0.7501 | train acc: 0.7490 | val loss: 1.3011 | val acc: 0.5432
Epoch 05 | train loss: 0.6578 | train acc: 0.7876 | val loss: 1.2736 | val acc: 0.5519
Epoch 06 | train loss: 0.5919 | train acc: 0.8084 | val loss: 1.3465 | val acc: 0.5658
Epoch 07 | train loss: 0.5358 | train acc: 0.8307 | val loss: 1.3270 | val acc: 0.5585
Epoch 08 | train loss: 0.5009 | train acc: 0.8415 | val loss: 1.4283 | val acc: 0.5452
Epoch 09 | train loss: 0.4632 | train acc: 0.8533 | val loss: 1.5169 | val acc: 0.5485
Epoch 10 | train loss: 0.4460 | train acc: 0.8603 | val loss: 1.4971 | val acc: 0.5445
Training done! Time taken: 11.574 seconds
token-indicator
Vocabulary size: 8399
Transforming documents...
Document transformation 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7060 | train acc: 0.3319 | val loss: 1.3266 | val acc: 0.5230
Epoch 02 | train loss: 1.2454 | train acc: 0.5551 | val loss: 1.2519 | val acc: 0.5489
Epoch 03 | train loss: 0.9850 | train acc: 0.6514 | val loss: 1.2553 | val acc: 0.5542
Epoch 04 | train loss: 0.8388 | train acc: 0.7133 | val loss: 1.2354 | val acc: 0.5622
Epoch 05 | train loss: 0.6973 | train acc: 0.7638 | val loss: 1.2971 | val acc: 0.5556
Epoch 06 | train loss: 0.6607 | train acc: 0.7775 | val loss: 1.3238 | val acc: 0.5595
Epoch 07 | train loss: 0.5970 | train acc: 0.8014 | val loss: 1.3556 | val acc: 0.5622
Epoch 08 | train loss: 0.5296 | train acc: 0.8284 | val loss: 1.4212 | val acc: 0.5609
Epoch 09 | train loss: 0.5063 | train acc: 0.8384 | val loss: 1.3945 | val acc: 0.5662
Epoch 10 | train loss: 0.4548 | train acc: 0.8447 | val loss: 1.4348 | val acc: 0.5649
Training done! Time taken: 10.670 seconds
token-indicator
Vocabulary size: 8413
Transforming documents...
Document transformation 

... (22 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.6576 | train acc: 0.3684 | val loss: 1.2909 | val acc: 0.5429
Epoch 02 | train loss: 1.0770 | train acc: 0.6341 | val loss: 1.2964 | val acc: 0.5376
Epoch 03 | train loss: 0.8279 | train acc: 0.7337 | val loss: 1.2679 | val acc: 0.5502
Epoch 04 | train loss: 0.7402 | train acc: 0.7836 | val loss: 1.2684 | val acc: 0.5489
Epoch 05 | train loss: 0.6934 | train acc: 0.8099 | val loss: 1.2914 | val acc: 0.5383
Epoch 06 | train loss: 0.5789 | train acc: 0.8432 | val loss: 1.2938 | val acc: 0.5456
Epoch 07 | train loss: 0.5511 | train acc: 0.8678 | val loss: 1.3026 | val acc: 0.5436
Epoch 08 | train loss: 0.5377 | train acc: 0.8688 | val loss: 1.3663 | val acc: 0.5369
Epoch 09 | train loss: 0.4868 | train acc: 0.8881 | val loss: 1.3583 | val acc: 0.5343
Epoch 10 | train loss: 0.4689 | train acc: 0.8881 | val loss: 1.3764 | val acc: 0.5376
Training done! Time taken: 45.061 seconds
bigrams-and-trigrams
Vocabulary size: 58621
Transforming documents...
Document transform

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.6791 | train acc: 0.3522 | val loss: 1.3031 | val acc: 0.5349
Epoch 02 | train loss: 1.1376 | train acc: 0.6017 | val loss: 1.2263 | val acc: 0.5768
Epoch 03 | train loss: 0.9088 | train acc: 0.7061 | val loss: 1.2342 | val acc: 0.5576
Epoch 04 | train loss: 0.7972 | train acc: 0.7550 | val loss: 1.2271 | val acc: 0.5848
Epoch 05 | train loss: 0.7352 | train acc: 0.7910 | val loss: 1.2308 | val acc: 0.5722
Epoch 06 | train loss: 0.6272 | train acc: 0.8314 | val loss: 1.2814 | val acc: 0.5542
Epoch 07 | train loss: 0.6001 | train acc: 0.8467 | val loss: 1.2663 | val acc: 0.5622
Epoch 08 | train loss: 0.5300 | train acc: 0.8618 | val loss: 1.3545 | val acc: 0.5236
Epoch 09 | train loss: 0.5276 | train acc: 0.8690 | val loss: 1.3382 | val acc: 0.5243
Epoch 10 | train loss: 0.4965 | train acc: 0.8862 | val loss: 1.3331 | val acc: 0.5436
Training done! Time taken: 164.093 seconds


In [24]:
print(encoder_accuracy)

{'token-indicator': np.float64(0.5912727736017327), 'bag-of-words': np.float64(0.5878138227091915), 'bigrams-and-trigrams': np.float64(0.594600161379369)}
